<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/06-adaptation/01-fine-tune-vs-rag-vs-prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adapting the Model: Fine-Tune vs RAG vs Prompt

**Goal:** Make the call an AI engineer is paid to make — when to *change the model* (fine-tune / LoRA) versus change its *inputs* (prompt, RAG) — and be able to defend it. Then, optionally, run a real LoRA fine-tune on a free GPU.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Setup

The concept sections (1–4) run on the free Groq API like every other notebook — the two cells below are all they need. The **optional LoRA appendix** at the end needs a GPU runtime and a different stack; it installs its own dependencies and is clearly fenced, so you can read this notebook end-to-end without ever running it.

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client.
2. **Load your API key.** Free key at [console.groq.com](https://console.groq.com/) (no credit card); in Colab add it via the **key icon** → **Add new secret** named exactly `GROQ_API_KEY`, notebook access on. Locally, set `GROQ_API_KEY` in your environment.

(Full walkthrough: [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Loads GROQ_API_KEY and returns a ready Groq client. Used only by the concept
# sections below; the optional LoRA appendix runs on a separate GPU stack.
client, MODEL = setup()

## Three ways to change a model's behavior

When a model doesn't do what you need, you have exactly three levers. In the order you should reach for them — cheapest and most reversible first:

1. **Prompting** — change the *instructions*. System prompt, few-shot examples, output format, tone. Zero training, instant iteration, no infrastructure.
2. **RAG** — change the *inputs*. Retrieve relevant facts at query time and put them in the prompt (all of section 03). The model's weights never change; you're feeding it better context.
3. **Fine-tuning** — change the *weights*. Continue training the model on your examples so the new behavior is baked in. **LoRA/QLoRA** is the modern, cheap way to do this (more below).

The single most common mistake in this space is reaching for lever 3 first — it's the most expensive, the slowest to iterate, and the one people assume is the "real" solution because it sounds like ML. The AI engineer's job is usually to talk a stakeholder *down* the list, not up it. This notebook is about knowing exactly when climbing to lever 3 is right.

## The decision, in one table

The instinct to internalize: **prompting and RAG change what the model *knows right now*; fine-tuning changes how the model *behaves by default*.** Knowledge → context. Behavior → weights.

| You want to change... | Reach for | Why |
|---|---|---|
| **Facts / knowledge** the model lacks (docs, private data, recent events) | **RAG** (or prompt) | Knowledge changes and must be cited; retrieval keeps it fresh and grounded. Fine-tuning bakes facts in as of training day — stale, uncitable, expensive to update. |
| **Format / structure** (always emit this JSON shape, this schema) | **Prompt** first; fine-tune if it must be bulletproof at scale | Prompting + tool-calling (section 01) covers most of it; fine-tuning removes the last few % of format drift on very high volume. |
| **Style / tone / persona** (sound like our brand, our support voice) | **Fine-tune (LoRA)** | Style is diffuse behavior, hard to fully specify in a prompt; a few hundred examples teach it better than paragraphs of instructions. |
| **A narrow, repetitive task** done cheaper/faster (classify, extract, rewrite at huge volume) | **Fine-tune a small model** | A fine-tuned small model can match a prompted big one on one narrow task — at a fraction of the per-call cost and latency. |
| **New skill / reasoning the base model just can't do** | **Fine-tune**, maybe — but validate hard | Rare for this role, and easy to overestimate; often a better base model or better prompting closes the gap first. |

Two rules that fall out of the table:

- **Fine-tuning is the wrong tool for knowledge.** This is the #1 misconception. "The model doesn't know our product docs → let's fine-tune on them" is almost always a mistake: you get an expensive model that has *memorized* docs it can't cite and can't update without retraining. That's RAG's job.
- **They compose.** Real systems often fine-tune for *style/format* AND use RAG for *knowledge* on the same model. It's not either/or.

## What LoRA / QLoRA actually are

Full fine-tuning updates *every* weight in the model — for a 7B model that's ~7 billion numbers, needing many tens of GB of GPU memory and a serious machine. That's what made fine-tuning inaccessible for years.

**LoRA (Low-Rank Adaptation)** is the trick that changed it. Instead of updating the big weight matrices, you *freeze* them and train tiny "adapter" matrices alongside — typically **<1% of the parameters**. The insight: the *change* you need is low-rank (expressible with far fewer numbers than the full matrix), so a small adapter captures it. You get most of full fine-tuning's quality at a fraction of the memory and storage.

- **LoRA** — freeze the base model, train small low-rank adapters. Adapters are a few MB; you can keep many per base model and swap them.
- **QLoRA** — LoRA on top of a **quantized** (4-bit) base model. Cuts memory further, enough to fine-tune a 7B model on a single free-tier Colab GPU.
- **PEFT** — HuggingFace's library (Parameter-Efficient Fine-Tuning) that implements LoRA/QLoRA and friends. This is the standard tooling today.

Why this matters for you: LoRA is now *mainstream*, not exotic. "We'll fine-tune a LoRA" is a normal sentence in this job, and adapters are cheap enough to be a real option — which makes knowing *when they're the wrong option* more valuable, not less. (Some inference providers, Groq included, can even **serve** LoRA adapters at inference time, so the trained adapter is deployable, not just a lab artifact.)

### Two ways to actually do it: hosted API vs train-it-yourself

There are two paths to a fine-tuned model, and most FDEs reach for the first:

**1. Hosted fine-tuning API** (OpenAI, Google, Bedrock, …) — upload a `.jsonl` of examples, call one method; the provider trains *and* hosts the result, and you call it like any other model. No GPU, no training loop. This is the common production path. OpenAI shape, as reference (**don't run — needs the SDK + a paid key**):

```python
# pip install openai  — reference only, not run here
from openai import OpenAI
client = OpenAI()

# training data: a .jsonl where each line is one chat example ->
#   {"messages": [{"role": "system", "content": "..."},
#                 {"role": "user", "content": "..."},
#                 {"role": "assistant", "content": "the target reply"}]}
f = client.files.create(file=open("train.jsonl", "rb"), purpose="fine-tune")

job = client.fine_tuning.jobs.create(training_file=f.id, model="gpt-4.1-mini-2025-04-14")
job = client.fine_tuning.jobs.retrieve(job.id)      # poll until status == "succeeded"
# then use it like any model:
# client.chat.completions.create(model=job.fine_tuned_model, messages=[...])
```

(Bedrock exposes the same idea for hosted open models — a create-customization-job call that yields a private fine-tuned/LoRA endpoint you invoke normally.)

**2. Train the adapter yourself** with `peft`/`trl` on your own GPU — full control, runs on open weights, no per-token markup on a hosted custom model, but you own the training *and* the serving. That's the optional appendix at the end of this notebook.

Either way the *decision* and the *evaluation* are identical — which is the whole point of this notebook. Whether you rent the training or run it, you still fine-tune only when prompt + RAG have demonstrably failed an eval, and you still prove it helped with the same harness from section 02 / section 04.

## Cost, and the argument you'll actually have

You will, at some point, be in a room where someone says *"can't we just fine-tune it?"* Here's the honest cost picture to answer with — not to shut it down, but to sequence it right:

| | Prompt / RAG | LoRA fine-tune |
|---|---|---|
| **Upfront cost** | ~none | a labeled dataset (hundreds–thousands of examples) + a GPU training run |
| **Iteration speed** | seconds — edit and rerun | hours — re-curate data, retrain, re-eval |
| **Updating knowledge** | change the retrieved docs | retrain (knowledge is frozen into weights) |
| **Per-call cost at scale** | pay for the context tokens every call | a fine-tuned *small* model can be much cheaper per call |
| **Can you cite sources?** | yes (RAG) | no |
| **Main risk** | prompt bloat, retrieval quality | overfitting, stale knowledge, an eval you didn't build |

The professional move is **prompt → RAG → fine-tune, and only advance when the cheaper lever demonstrably fails an eval.** Fine-tuning without an eval (section 04) is how teams spend a GPU week to make a model *feel* better and *measurably* worse. If you take one thing from this notebook into an interview: you can explain *why you'd usually reach for RAG first, and the specific conditions under which you'd fine-tune anyway* (style/format/latency at volume — rarely knowledge).

In [ ]:
# A concept check you CAN run on Groq: does the base model already do the task well
# enough with a good prompt? Answering this honestly is the gate before any fine-tune.
# Here: a formatting/style task, the kind people reach to fine-tune for.
examples = [
    "ticket: wifi keeps dropping in the west conference room since tuesday",
    "ticket: cant log into the vpn, mfa code never arrives",
]
SYSTEM = ('You are a support triage bot. For each ticket reply with ONE line: '
          '"<PRIORITY> | <CATEGORY> | <one-line summary>". '
          'PRIORITY in {P1,P2,P3}. CATEGORY in {network,auth,hardware,other}. Nothing else.')

for t in examples:
    r = client.chat.completions.create(
        model=MODEL, max_tokens=60,
        messages=[{'role': 'system', 'content': SYSTEM},
                  {'role': 'user', 'content': t}],
    )
    print(r.choices[0].message.content.strip())

# If a prompt like this already nails the format across your real ticket distribution,
# you do NOT need to fine-tune for format. Fine-tuning earns its place only when the
# prompt fails often enough, at high enough volume, that baking it in pays for itself.
print('\n-> Prompt got the format. Now ask: does it hold on 500 real tickets? '
      'That eval — not a hunch — is what justifies (or kills) a fine-tune.')

---

## Optional appendix: a real LoRA fine-tune (GPU runtime required)

Everything above runs on Groq. **This section does not.** Training weights needs a GPU and a different stack (`peft`, `transformers`, `trl`, `bitsandbytes`) — none of which touch Groq. Treat it as a lab you *can* run to make LoRA concrete, not part of the Groq path.

**To run it:** in Colab, **Runtime → Change runtime type → T4 GPU**, then run the cells below. On a CPU runtime the guard cell will stop you with a clear message instead of crashing. This trains a tiny LoRA adapter on a small open model so it finishes in minutes on a free T4 — it's a *mechanics* demo, not a quality result.

If you're just here for the judgment (the point of this notebook), you can stop at the line above — you already have it.

In [ ]:
# GUARD: this appendix needs a GPU. Stop early and clearly on CPU rather than
# installing a heavy stack and failing deep in a training loop.
import torch
assert torch.cuda.is_available(), (
    'No GPU detected. This appendix requires one: Runtime -> Change runtime type '
    '-> T4 GPU, then re-run. (The concept sections above do not need this.)'
)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Heavy, GPU-only deps. Not part of the Groq stack; installed here on purpose.
%pip install -q -U transformers peft trl datasets bitsandbytes accelerate

In [ ]:
# A tiny LoRA fine-tune, start to finish, on a small open model.
# Task: teach a fixed reply STYLE (terse, prefixed) from a handful of examples --
# style is exactly what fine-tuning is good at and prompting struggles to pin down.
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

BASE = 'HuggingFaceTB/SmolLM2-135M-Instruct'  # tiny: trains in minutes on a T4

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')

# Toy dataset: teach a house style -> every answer starts "ACME> " and stays terse.
raw = [
    ("How do I reset my password?", "ACME> Settings > Security > Reset password. Done."),
    ("Is the API down?", "ACME> Check status.acme.io. If red, we're on it."),
    ("How do I export my data?", "ACME> Settings > Data > Export. You'll get a CSV by email."),
    ("Can I get a refund?", "ACME> Yes within 30 days. Reply here with your order id."),
] * 16  # repeat so the tiny model actually picks up the pattern
ds = Dataset.from_list([
    {'text': f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"}
    for q, a in raw
])

peft_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                      target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM')

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    peft_config=peft_cfg,
    args=SFTConfig(output_dir='lora-out', num_train_epochs=3,
                   per_device_train_batch_size=4, learning_rate=2e-4,
                   logging_steps=5, report_to='none', max_length=128),
)
trainer.train()
print('\nTrained a LoRA adapter with', sum(p.numel() for p in model.parameters() if p.requires_grad),
      'trainable params (vs', sum(p.numel() for p in model.parameters()), 'total).')

In [ ]:
# Before/after the adapter, same prompt. The trained adapter should nudge replies
# toward the "ACME> ..., terse" house style the base model doesn't use by default.
def reply(prompt):
    msgs = [{'role': 'user', 'content': prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(ids, max_new_tokens=40, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

q = 'How do I change my email address?'
with model.disable_adapter():          # base model, adapter off
    print('BASE :', reply(q))
print('LoRA :', reply(q))              # adapter on

print('\nThis is a mechanics demo on a 135M model and a toy dataset -- do not read '
      'quality into it. The point: <1% of params, a few minutes, style learned. '
      'Whether it BEATS a good prompt is an eval question (section 04), every time.')

## Exercises

1. **Argue it in an interview.** In 3–4 sentences each, answer: (a) *"A customer wants the model to know their internal wiki — fine-tune or RAG?"* (b) *"They want every answer in their brand voice — fine-tune or prompt?"* (c) *"They classify 2M tickets/day and GPT-4-class latency is too slow — what do you propose?"* These are real FDE screening questions.
2. **Kill a fine-tune with a prompt.** Take the house-style task from the appendix and try to get the *base* model (adapter off, or any Groq model) to match the "ACME> terse" style with prompting + few-shot alone. How close can you get? This is the honest first step before any fine-tune.
3. **Cost the decision.** For a narrow classification task at 1M calls/day, estimate monthly cost of (a) prompting a large model vs (b) a fine-tuned small model (reuse the cost math from `01-model-apis/04-context-and-caching`). At what volume does the fine-tune's upfront cost pay back?
4. **(GPU) Vary the LoRA rank.** Re-run the appendix with `r=1` and `r=64`. Does higher rank change the learned style or just the trainable-param count? Relate it to the low-rank insight: how small a `r` still captures this task?